# 002: False Green with Full Coverage and the No-Recovery Theorem

**Local Green, Global Red** — Notebook 2 of 3

**Core Topic:** Theorem 3 (an explicit 6-site system with 100% interface coverage that reports All-Green for any severity $R > 0$) and Theorem 4 (the mathematical impossibility of any finite recovery function).

**AuthorityEffect:** None.


In [1]:
import sys
from pathlib import Path
from fractions import Fraction
import numpy as np

# Dynamically locate and import sibling code directory
nb_dir = Path.cwd().resolve()
candidates = [
    nb_dir.parent / "code",
    nb_dir / "code",
] + [p / "code" for p in nb_dir.parents if (p / "code" / "code_core.py").is_file()]

for c in candidates:
    if c.is_dir() and (c / "lgr_replay.py").is_file():
        if str(c) not in sys.path:
            sys.path.insert(0, str(c))
        break

from lgr_replay import (
    three_patch_raw_norm, three_patch_tallies, three_patch_full_coverage,
    defeat_recovery, THREE_PATCH_EDGES
)
from code_witnesses import ThreePatchSystem
from code_theorems import replay_theorem_3_three_patch, replay_theorem_4_no_recovery_modulus

print("[OK] Exact replay and witness modules loaded successfully!")

[OK] Exact replay and witness modules loaded successfully!


## 1. Complete Coverage Does Not Guarantee Soundness

Common intuition in software testing and AI evaluation assumes that if *every interface and contract is monitored*, serious failures cannot remain hidden.

**Theorem 3** disproves this intuition with an explicit, finite construction:
- **6 Sites:**
  - 3 component sites: $u_0, u_1, u_2$
  - 3 overlap sites: $x_{01}, x_{12}, x_{20}$
- **6 Directed Interfaces:**
  - $e_{01}^0 = (x_{01}, u_0)$, $e_{01}^1 = (x_{01}, u_1)$
  - $e_{12}^1 = (x_{12}, u_1)$, $e_{12}^2 = (x_{12}, u_2)$
  - $e_{20}^2 = (x_{20}, u_2)$, $e_{20}^0 = (x_{20}, u_0)$
- **State Spaces:** $\mathbb{R}^3$ with $\ell^1$ norm, identity translations $r_e = I_3$.
- **3 Evaluators:** Evaluator $\alpha \in \{0, 1, 2\}$ watches both interfaces into $u_\alpha$ using measurement $q^\alpha$ that deletes coordinate $\alpha$.

### Why Overlap Sites?
On a simple 3-site triangle $u_0, u_1, u_2$, interfaces in opposite directions have defects $s(u_1) - s(u_0) = -(s(u_0) - s(u_1))$. Being negatives of each other, forcing them to lie along different coordinate axes forces them to vanish.
The overlap construction realizes the required hidden directions without the opposite-edge constraint; it does not make all six interface defects independent.

In [2]:
# Verify that every single interface is watched
print(f"Total interfaces in three-patch system: {len(THREE_PATCH_EDGES)}")
for edge in THREE_PATCH_EDGES:
    print(f"Interface {edge} is monitored by Evaluator {edge[0]}")

assert three_patch_full_coverage()
print("\n[CONFIRMED] Complete interface coverage: 100% of interfaces are watched!")

Total interfaces in three-patch system: 6
Interface (0, 0) is monitored by Evaluator 0
Interface (0, 1) is monitored by Evaluator 0
Interface (1, 0) is monitored by Evaluator 1
Interface (1, 1) is monitored by Evaluator 1
Interface (2, 0) is monitored by Evaluator 2
Interface (2, 1) is monitored by Evaluator 2

[CONFIRMED] Complete interface coverage: 100% of interfaces are watched!


## 2. The Arbitrary-Severity Scaling Family $s_R$

For any target severity $R > 0$, define the assignment:
- $s_R(u_i) = \lambda e_i$ (where $e_i$ is the $i$-th standard basis vector in $\mathbb{R}^3$)
- $s_R(x_{ij}) = 0$ on all overlap sites
- Scale $\lambda = R / 6$

Across each interface into $u_i$, the defect is:
$$\Delta(s_R)_e = r_e s_R(u_i) - s_R(x_{ij}) = \lambda e_i - 0 = \lambda e_i$$

Evaluator $i$ watches the two interfaces into $u_i$ through $q^i$, which projects away coordinate $i$:
$$q^i(\lambda e_i) = 0!$$

Therefore:
- **Evaluator Scores:** $J^0 = J^1 = J^2 = 0$ (**All-Green!**)
- **Global Inconsistency:** $I(s_R) = 6 \cdot \|\lambda e_i\|_1 = 6 |\lambda| = R$ (**Global Red at severity $R$!**)

Let's verify this in exact rational arithmetic:

In [3]:
test_severities = [1, 6, 42, 1000, 1000000]

print(f"{'Target R':>12} | {'lambda':>12} | {'Evaluator Tallies (J0, J1, J2)':>32} | {'True Inconsistency I(s)':>24}")
print("-" * 90)

for R in test_severities:
    lam = Fraction(R, 6)
    tallies = three_patch_tallies(lam)
    raw_I = three_patch_raw_norm(lam)
    
    print(f"{R:>12} | {str(lam):>12} | {str(tallies):>32} | {str(raw_I):>24}")
    assert tallies == (Fraction(0), Fraction(0), Fraction(0)), "Scores must be zero!"
    assert raw_I == Fraction(R), f"Inconsistency must equal {R}!"

print("\n[PASSED] In exact arithmetic, evaluator scores remain (0, 0, 0) while I(s) equals R exactly!")

    Target R |       lambda |   Evaluator Tallies (J0, J1, J2) |  True Inconsistency I(s)
------------------------------------------------------------------------------------------
           1 |          1/6 | (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) |                        1
           6 |            1 | (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) |                        6
          42 |            7 | (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) |                       42
        1000 |        500/3 | (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) |                     1000
     1000000 |     500000/3 | (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) |                  1000000

[PASSED] In exact arithmetic, evaluator scores remain (0, 0, 0) while I(s) equals R exactly!


## 3. Theorem 4: The No-Recovery Theorem

Can we fix this by post-processing evaluator scores? E.g., taking max scores, weighted sums, or nonlinear aggregation?

**Theorem 4 (No Recovery Modulus).** There is **no finite-valued function** $f: \mathbb{R}_{\ge 0} \to \mathbb{R}_{\ge 0}$ such that:
$$I(s) \le f\left(\max_\alpha J^\alpha(s)\right) \quad \forall s$$
and no joint function $F: \mathbb{R}_{\ge 0}^3 \to \mathbb{R}_{\ge 0}$ such that:
$$I(s) \le F(J^0(s), J^1(s), J^2(s)) \quad \forall s$$

*Proof.* Suppose such a function $f$ exists. Since $J^\alpha(s_R) = 0$ for all $\alpha$, we have $\max_\alpha J^\alpha(s_R) = 0$.
The bound would require:
$$R = I(s_R) \le f(0) \quad \forall R > 0$$
Choosing $R = f(0) + 1$ yields:
$$f(0) + 1 \le f(0) \implies 1 \le 0,$$
an immediate contradiction.

Let's compute explicit witness adversaries for arbitrary proposed bounds $f(0)$:

In [4]:
# Suppose an auditor claims: 'If all evals report 0, inconsistency is at most f0'
proposed_bounds = [0, 10, 1000, 999999]

for f0 in proposed_bounds:
    lam, tallies, raw_I = defeat_recovery(Fraction(f0))
    print(f"Claimed bound f(0) = {f0:>6} -> Adversarial lambda = {float(lam):.3f}")
    print(f"  Result: Tallies = {tallies} (All-Green)")
    print(f"  Result: True Inconsistency I(s) = {raw_I} > {f0} (Claim defeated!)\n")
    assert raw_I > f0

# Test numerical replay functions from code_theorems
success_t3, true_I, scores = replay_theorem_3_three_patch(R=60.0)
assert success_t3
print(f"[REPLAY] Theorem 3 numerical test: R=60.0, True I={true_I}, Evaluator scores={scores}")

candidate_f = lambda j: 50.0 + 10.0 * j
success_t4, i_val, bound_val = replay_theorem_4_no_recovery_modulus(candidate_f)
assert success_t4
print(f"[REPLAY] Theorem 4 numerical test: Bound f(0)={bound_val}, Defeated by I(s)={i_val}")
print("[OK] The displayed numerical witness checks passed.")

Claimed bound f(0) =      0 -> Adversarial lambda = 0.167
  Result: Tallies = (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) (All-Green)
  Result: True Inconsistency I(s) = 1 > 0 (Claim defeated!)

Claimed bound f(0) =     10 -> Adversarial lambda = 1.833
  Result: Tallies = (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) (All-Green)
  Result: True Inconsistency I(s) = 11 > 10 (Claim defeated!)

Claimed bound f(0) =   1000 -> Adversarial lambda = 166.833
  Result: Tallies = (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) (All-Green)
  Result: True Inconsistency I(s) = 1001 > 1000 (Claim defeated!)

Claimed bound f(0) = 999999 -> Adversarial lambda = 166666.667
  Result: Tallies = (Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)) (All-Green)
  Result: True Inconsistency I(s) = 1000000 > 999999 (Claim defeated!)

[REPLAY] Theorem 3 numerical test: R=60.0, True I=60.0, Evaluator scores={'evaluator_0': 0.0, 'evaluator_1': 0.0, 'evaluator_2': 0.0}
[REPLAY] Theorem 4 numerical test: Bound

### Takeaway from Notebook 2
1. Complete interface coverage is **not sufficient** for soundness.
2. A nonzero realizable defect in the product of watcher kernels yields arbitrarily large invisible defect mass by scaling, when assignments are unrestricted.
3. In Notebook 3, we explore the **Master Soundness Criterion** and discover why defect coupling can restore soundness even when evaluators have massive blind spots!